# FLAG 2027 - Notebook 22: how many heads? (ensemble-size curve for the frozen system A recipe)

On the dev set, 3 seeds -> 7 seeds of system A gained 0.9 EER (28.83 -> 27.91), almost all in the Bengali cells. This notebook measures, in people-disjoint cross-validation, how the EER keeps falling with 1, 3, 5, 7, 11, 15, 21 heads, and whether identity-bagging (each head sees a random 85% of the training identities) adds useful diversity.

Same recipe, data and features as the frozen system A (NB20): CLIP | ECAPA+WavLM+ResNet293+ReDimNet2+audeering; challenge + MAV v1/v2/v3 + VoxCeleb2; per-dataset normalisation; batch 64x8 with 75% in-domain ids; cosine LR; 18 epochs. Only cached features are used, so it is head training only: ~1 min per head, 3 folds x 2 variants x 21 seeds ~ 2 h. Resumable (per-head scores cached on Drive). Paste the **SUMMARY**.

Decision rule: if the 21-head ensemble beats the 7-head one by >= 0.3 on the fold average (both std and gender), refreeze NB20 with N_SEEDS = 21. If bagging beats plain at equal size by >= 0.3, add bagging to NB20.

In [ ]:
# Cell 1 - Drive, GPU
from google.colab import drive
drive.mount('/content/drive')
import torch, shutil
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', dev)

In [ ]:
# Cell 2 - features, labels, folds, model (identical to NB20 Cell 2/3)
import os, json, time, zipfile, glob
import numpy as np, pandas as pd
import torch.nn as nn, torch.nn.functional as Fn
from sklearn.metrics import roc_curve
from tqdm.auto import tqdm
BASE = '/content/drive/MyDrive'; PROJECT = os.path.join(BASE, 'FLAG2027')
FEATDIR = os.path.join(PROJECT, 'features'); RESDIR = os.path.join(PROJECT, 'results')
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
LOCAL = '/content/flag_small'; os.makedirs(LOCAL, exist_ok=True)
if not os.path.exists(os.path.join(LOCAL, 'train.done')):
    with zipfile.ZipFile(os.path.join(ZIP_DIR, 'train_set.zip')) as zf:
        zf.extractall(LOCAL, members=[m for m in zf.namelist() if m.endswith(('.csv', '.txt'))])
    open(os.path.join(LOCAL, 'train.done'), 'w').close()
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)
tv = glob.glob(LOCAL + '/**/train_English_voices.csv', recursive=True)[0]; tl = glob.glob(LOCAL + '/**/train_English.txt', recursive=True)[0]
Xv, y = read_feat(tv, 192); y = y.astype(int)
CHF = {'ecapa': Xv}
for name in ['clip', 'wavlm_sv', 'r293', 'redim', 'audeer']:
    CHF[name] = np.load(os.path.join(FEATDIR, name + '.npz'))['train']
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz'))); EXTRA = dict(np.load(os.path.join(FEATDIR, 'mavceleb_extra.npz')))
for name in ['r293', 'redim', 'audeer']:
    d = np.load(os.path.join(FEATDIR, name + '.npz')); MAV[name] = d['mav']
    if name != 'r293':
        EXTRA[name] = d['extra']
VOX = {k: [] for k in ['spk', 'clip', 'ecapa', 'wavlm_sv', 'r293', 'redim', 'audeer']}
for tag in ['vox2_test_mp4', 'vox2_mp4_1']:
    d = dict(np.load(os.path.join(FEATDIR, 'vox_' + tag + '.npz')))
    d['r293'] = np.load(os.path.join(FEATDIR, 'vox_%s_r293.npy' % tag)); d['redim'] = np.load(os.path.join(FEATDIR, 'vox_%s_redim.npy' % tag))
    d['audeer'] = np.load(os.path.join(FEATDIR, 'vox_%s_audeer.npz' % tag))['h']
    for k in VOX:
        VOX[k].append(d[k])
VOX = {k: np.concatenate(v) for k, v in VOX.items()}
N_CH, N_MAV, N_EX, N_VOX = len(y), len(MAV['spk']), len(EXTRA['spk']), len(VOX['spk'])
ext_spk = list(MAV['spk']) + list(EXTRA['spk']) + list(VOX['spk'])
ext_lab = {s: 1000 + i for i, s in enumerate(sorted(set(ext_spk)))}
Y_ALL = np.concatenate([y, np.array([ext_lab[s] for s in ext_spk])])
D_ALL = np.r_[np.zeros(N_CH, int), np.ones(N_MAV + N_EX, int), 2 * np.ones(N_VOX, int)]
VOICE = ['ecapa', 'wavlm_sv', 'r293', 'redim', 'audeer']
F_RAW = np.concatenate([l2(CHF['clip']), l2(MAV['clip']), l2(EXTRA['clip']), l2(VOX['clip'])]).astype(np.float32)
V_RAW = np.concatenate([np.concatenate([l2(src[n]) for n in VOICE], 1) for src in (CHF, MAV, EXTRA, VOX)]).astype(np.float32)
note('rows: challenge %d | MAV %d | VoxCeleb %d | identities %d' % (N_CH, N_MAV + N_EX, N_VOX, len(np.unique(Y_ALL))))
# gender of the challenge ids (for same-gender validation pairs) + the usual gender-stratified 5 folds (seed 42)
train_lines = [l.split() for l in open(tl) if l.strip()]; cls2spk = {int(l[5]): l[4] for l in train_lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y); id_gender = {c: spk2g[cls2spk[c]] for c in ids}
def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
def make_pairs(idx, n=2000, same_gender_neg=False, seed=0):
    rng = np.random.default_rng(seed); spk_ = np.unique(y[idx]); rows_ = {s: idx[y[idx] == s] for s in spk_}
    V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk_); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[s])); L.append(1)
        s = rng.choice(spk_)
        cand = [t for t in spk_ if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])] or [t for t in spk_ if t != s]
        t = rng.choice(cand); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)
K = 5; rng = np.random.default_rng(42); folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)
FOLD_DATA = []
for k in range(K):
    val_ids = set(folds[k]); va = np.where([c in val_ids for c in y])[0]; tr = np.where([c not in val_ids for c in y])[0]
    FOLD_DATA.append((tr, [make_pairs(va, seed=k), make_pairs(va, same_gender_neg=True, seed=100 + k)]))
_PROTO = {}
def max_sep_prototypes(n):
    if n in _PROTO:
        return _PROTO[n]
    P = np.array([[1.0, -1.0]])
    for k in range(2, n):
        Q = np.zeros((k, k + 1)); Q[0, 0] = 1.0; Q[0, 1:] = -1.0 / k; Q[1:, 1:] = np.sqrt(1 - 1.0 / k ** 2) * P; P = Q
    _PROTO[n] = torch.tensor(P.T, dtype=torch.float32); return _PROTO[n]
class Branch(nn.Module):
    def __init__(self, d_in, d=128):
        super().__init__(); self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)
class FOP(nn.Module):
    def __init__(self, n_class, d_face, d_voice, d=128):
        super().__init__()
        self.face, self.voice = Branch(d_face, d), Branch(d_voice, d)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.cls = nn.Linear(d, n_class - 1); self.register_buffer('P', max_sep_prototypes(n_class))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        f, v = self.face(xf), self.voice(xv); return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv):
        ft, vt, f, v = self.embed(xf, xv); a = torch.sigmoid(self.att(torch.cat([f, v], 1))); fused = ft * a + (1 - a) * vt
        return fused, self.cls(fused) @ self.P.t()
def opl_loss(feat, lab):
    feat = Fn.normalize(feat, dim=1); dot = feat @ feat.t(); same = (lab[:, None] == lab[None, :]).float()
    pos = same - torch.eye(len(lab), device=feat.device); neg = 1 - same
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)
@torch.no_grad()
def head_score(model, xf, xv):
    model.eval(); mf, sf_, mv, sv = model.norm; ft, vt, _, _ = model.embed((xf - mf) / sf_, (xv - mv) / sv)
    return -(ft - vt).norm(dim=1).cpu().numpy()
R18 = pd.read_csv(os.path.join(RESDIR, 'nb18_cv.csv')); EPOCHS = int(np.median(R18[R18['variant'] == 'V1_batch64_cosine']['best_ep']))
note('recipe A: batch 64x8, 75%% in-domain, cosine LR, %d epochs (median best epoch of NB18 V1)' % EPOCHS)

In [ ]:
# Cell 3 - train 21 heads per fold and variant, cache the validation scores of every head
P_ID, K_CLIP, ALPHA, LR, STEPS, IN_FRAC = 64, 8, 2.0, 1e-4, 60, 0.75
N_HEADS, FOLDS, VARIANTS = 21, [0, 1, 2], {'plain': 1.0, 'bag85': 0.85}
def train_head(tr_idx, seed, keep_frac=1.0):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    all_tr = np.concatenate([tr_idx, np.where(D_ALL > 0)[0]])
    if keep_frac < 1.0:                                           # identity bagging: drop a random 15% of TRAINING identities for this head
        tr_cls = np.unique(Y_ALL[all_tr]); keep = set(rng.choice(tr_cls, int(round(keep_frac * len(tr_cls))), replace=False))
        all_tr = all_tr[np.isin(Y_ALL[all_tr], list(keep))]
    F, V = F_RAW.copy(), V_RAW.copy()
    for d in np.unique(D_ALL):
        trd = all_tr[D_ALL[all_tr] == d]; rows = np.where(D_ALL == d)[0]
        F[rows] = (F_RAW[rows] - F_RAW[trd].mean(0)) / (F_RAW[trd].std(0) + 1e-6); V[rows] = (V_RAW[rows] - V_RAW[trd].mean(0)) / (V_RAW[trd].std(0) + 1e-6)
    XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev); tr = torch.tensor(all_tr, device=dev)
    cls_in = np.zeros(len(cl), bool); cls_in[np.unique(ytr_np[D_ALL[all_tr] < 2])] = True
    in_cls, out_cls = np.where(cls_in)[0], np.where(~cls_in)[0]; n_in = int(round(P_ID * IN_FRAC))
    pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    model = FOP(len(cl), XF.shape[1], XV.shape[1]).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6); mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01); sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100 * STEPS)
    for ep in range(EPOCHS):
        model.train()
        for _ in range(STEPS):
            chosen = np.r_[rng.choice(in_cls, n_in, replace=False), rng.choice(out_cls, P_ID - n_in, replace=False)]
            j = torch.tensor(np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]), device=dev)
            fused, logits = model((XF[tr[j]] - mf) / sf_, (XV[tr[j]] - mv) / sv)
            loss = Fn.cross_entropy(logits, ytr[j]) + ALPHA * opl_loss(fused, ytr[j]); opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return model, XF, XV
CACHE22 = os.path.join(RESDIR, 'nb22_scores'); os.makedirs(CACHE22, exist_ok=True)
t0 = time.time(); n_done = 0
for k in FOLDS:
    tr_idx, vsets = FOLD_DATA[k]
    for var, frac in VARIANTS.items():
        for seed in range(1, N_HEADS + 1):
            p = os.path.join(CACHE22, 'f%d_%s_s%02d.npz' % (k, var, seed))
            if os.path.exists(p):
                continue
            model, XF, XV = train_head(tr_idx, 1000 * k + seed, frac)
            np.savez(p, std=head_score(model, XF[vsets[0][1]], XV[vsets[0][0]]), gen=head_score(model, XF[vsets[1][1]], XV[vsets[1][0]]))
            n_done += 1; del model, XF, XV; torch.cuda.empty_cache()
            if n_done % 5 == 0:
                print('fold %d %-6s seed %2d done | %.0f min' % (k, var, seed, (time.time() - t0) / 60))
print('all heads trained')

In [ ]:
# Cell 4 - ensemble-size curve (EER of the raw mean of the first k head scores), per fold and averaged
SIZES = [1, 3, 5, 7, 11, 15, 21]
rows = []
for k in FOLDS:
    _, vsets = FOLD_DATA[k]; L_std, L_gen = vsets[0][2], vsets[1][2]
    for var in VARIANTS:
        S = [np.load(os.path.join(CACHE22, 'f%d_%s_s%02d.npz' % (k, var, s))) for s in range(1, N_HEADS + 1)]
        singles = [(eer(L_std, d['std']) + eer(L_gen, d['gen'])) / 2 for d in S]
        for n in SIZES:
            e_std = eer(L_std, np.mean([d['std'] for d in S[:n]], 0)); e_gen = eer(L_gen, np.mean([d['gen'] for d in S[:n]], 0))
            rows.append(dict(fold=k, variant=var, heads=n, std=e_std, gender=e_gen, overall=(e_std + e_gen) / 2))
        rows.append(dict(fold=k, variant=var, heads=0, std=np.nan, gender=np.nan, overall=np.mean(singles), single_sd=np.std(singles)))
R = pd.DataFrame(rows); R.to_csv(os.path.join(RESDIR, 'nb22_curve.csv'), index=False)
tab = R[R.heads > 0].groupby(['variant', 'heads'])[['std', 'gender', 'overall']].mean().round(2)
note('ENSEMBLE-SIZE CURVE (mean over folds %s; CV EER)' % FOLDS); note(tab.to_string())
sd = R[R.heads == 0].groupby('variant')['single_sd'].mean().round(2)
note('single-head EER spread (sd across heads): ' + ', '.join('%s %.2f' % (v, s) for v, s in sd.items()))
p7, p21 = tab.loc[('plain', 7), 'overall'], tab.loc[('plain', 21), 'overall']; b7, b21 = tab.loc[('bag85', 7), 'overall'], tab.loc[('bag85', 21), 'overall']
note('plain: 7 heads %.2f -> 21 heads %.2f (gain %.2f) | bag85: 7 heads %.2f -> 21 heads %.2f | best overall: %s' % (p7, p21, p7 - p21, b7, b21, tab['overall'].idxmin()))
print('=' * 70); print('SUMMARY  (copy everything between the lines and paste to Claude)'); print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)